# OpenAPI Specification JSON Schema from Online Documentation

While most of the public APIs provide OpenAPI JSON schema by default, some of them don't. In this case, we can use the online documentation with LLMs, like ChatGPT, to extract the JSON schema. This notebook demonstrates how to extract the JSON schema from the online documentation of the API.

## Bland AI API Documentation
<!-- ![title](assets/bland-ai-logo.png) -->

Bland is a platform for AI phone calling. Using our API, you can easily send or receive phone calls with a programmable voice agent.

![title](assets/bland-ai-functionality.png)

The API Documentation is here: [Bland.ai API Documentation](https://docs.bland.ai/api-v1/post/calls-simple)


## Convert Online Documentation to OpenAPI JSON Schema

For this example, let us convert `Send Call` API from Bland AI to OpenAPI JSON schema. The cURL request for the API specified in the documentation is as follows:

```bash
curl --request POST \
  --url https://api.bland.ai/v1/calls \
  --header 'Content-Type: application/json' \
  --header 'authorization: <authorization>' \
  --data '{
  "phone_number": "<string>",
  "task": "<string>",
  "pathway_id": "<string>",
  "start_node_id": "<string>",
  "voice": "<string>",
  "background_track": "<string>",
  "first_sentence": "<string>",
  "wait_for_greeting": true,
  "block_interruptions": true,
  "interruption_threshold": 123,
  "model": "<string>",
  "temperature": 123,
  "keywords": [
    "<string>"
  ],
  "pronunciation_guide": [
    {}
  ],
  "transfer_phone_number": "<string>",
  "transfer_list": {},
  "language": "<string>",
  "timezone": "<string>",
  "request_data": {},
  "tools": [
    {}
  ],
  "dynamic_data": [
    {
      "dynamic_data[i].response_data": [
        {}
      ]
    }
  ],
  "start_time": "<string>",
  "voicemail_message": "<string>",
  "voicemail_action": {},
  "retry": {},
  "max_duration": 123,
  "record": true,
  "from": "<string>",
  "webhook": "<string>",
  "webhook_events": [
    "<string>"
  ],
  "metadata": {},
  "summary_prompt": "<string>",
  "analysis_prompt": "<string>",
  "analysis_schema": {},
  "answered_by_enabled": true
}'
```

In [38]:
from slackagents.llms.openai import OpenAILLM, BaseLLMConfig
config = BaseLLMConfig(model="gpt-4o", max_tokens=2048)
llm = OpenAILLM(config)

prompt = """\
Write OpenAPI specification json file for this api call. Only include the json file. Include base url into servers field, and the endpoint and parameters into paths field.
```bash
curl --request POST \
  --url https://api.bland.ai/v1/calls \
  --header 'Content-Type: application/json' \
  --header 'authorization: <authorization>' \
  --data '{
  "phone_number": "<string>",
  "task": "<string>",
  "pathway_id": "<string>",
  "start_node_id": "<string>",
  "voice": "<string>",
  "background_track": "<string>",
  "first_sentence": "<string>",
  "wait_for_greeting": true,
  "block_interruptions": true,
  "interruption_threshold": 123,
  "model": "<string>",
  "temperature": 123,
  "keywords": [
    "<string>"
  ],
  "pronunciation_guide": [
    {}
  ],
  "transfer_phone_number": "<string>",
  "transfer_list": {},
  "language": "<string>",
  "timezone": "<string>",
  "request_data": {},
  "tools": [
    {}
  ],
  "dynamic_data": [
    {
      "dynamic_data[i].response_data": [
        {}
      ]
    }
  ],
  "start_time": "<string>",
  "voicemail_message": "<string>",
  "voicemail_action": {},
  "retry": {},
  "max_duration": 123,
  "record": true,
  "from": "<string>",
  "webhook": "<string>",
  "webhook_events": [
    "<string>"
  ],
  "metadata": {},
  "summary_prompt": "<string>",
  "analysis_prompt": "<string>",
  "analysis_schema": {},
  "answered_by_enabled": true
}'\
"""

messages = [
    {"role": "user", "content": prompt}
]

response = llm.chat_completion(messages)

Users can save the response as a JSON file. The JSON file can be used to generate the tool in the next step.

In [ ]:
from IPython.display import display, Markdown
import json
display(Markdown(response))
json_string = response.strip('```json').strip('```').strip()
data = json.loads(json_string)
with open("assets/bland_ai.json", "w") as f:
    json.dump(data, f, indent=4)

## OpenAPITool from JSON Schema
We can load the json file and use it to define an agent tool using `OpenAPITool` class.

In [ ]:
import os
from slackagents.tools.openapi_tool import OpenAPITool, AuthType

os.environ["BLANDAI_API_KEY_NAME"] = "authorization"
os.environ["BLANDAI_API_KEY_VALUE"] = "bland_ai_api_key"
os.environ["BLANDAI_API_KEY_IN"] = "header"
auth_type = AuthType.API_KEY

with open("assets/bland_ai.json", "r") as f:
    openapi_spec = json.load(f)

tool = OpenAPITool(
    name="bland_ai",
    openapi_spec=openapi_spec,
    auth_type=AuthType.API_KEY,
)
print(json.dumps(tool.info, indent=4))